# 01_01 - EnVi-Tech-Reasoning Data Collection

## Mục tiêu

Notebook này thực hiện bước **Data Collection** cho source:

**EnVi-Tech-Reasoning-SFT**

Các nhiệm vụ:

1. Kiểm tra môi trường
2. Xác định project root
3. Cấu hình source
4. Download dataset
5. Convert dữ liệu sang DataFrame
6. Inspect raw data
7. Thống kê raw data
8. Xác định technology candidate
9. Tổng hợp audit summary
10. Lưu raw JSONL
11. Lưu raw Parquet
12. Lưu audit summary
13. Lưu metadata
14. Final verification
15. Ghi trạng thái notebook

> Lưu ý:
> - Notebook này **không cleaning dữ liệu**
> - Không deduplication
> - Không train/validation/test split
> - Không overwrite raw data
> - Technology candidate **không đồng nghĩa** với usable IT corpus
> - `usable_count` chỉ được xác định sau các bước kiểm tra/audit tiếp theo

In [1]:
import sys
import os
from pathlib import Path

print("Python:", sys.version)
print("Working directory:", os.getcwd())

Python: 3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
Working directory: C:\Users\ADMIN\ENVI-IT-MT\notebooks\01_data_collection


In [2]:
import datasets
import pandas as pd
import httpx

print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("httpx:", httpx.__version__)

datasets: 5.0.1
pandas: 3.0.5
httpx: 0.28.1


In [3]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path) -> Path:
    """
    Tìm project root bằng cách kiểm tra:
    - data/
    - notebooks/ hoặc notebook/
    """
    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        if (
            (path / "data").is_dir()
            and (
                (path / "notebooks").is_dir()
                or (path / "notebook").is_dir()
            )
        ):
            return path

    raise FileNotFoundError(
        "Không tìm thấy project root. "
        "Hãy kiểm tra lại vị trí notebook."
    )


PROJECT_ROOT = find_project_root(CURRENT_DIR)

print("Project root:")
print(PROJECT_ROOT)

Project root:
C:\Users\ADMIN\ENVI-IT-MT


In [4]:
SOURCE_NAME = "EnVi-Tech-Reasoning-SFT"

SOURCE_SHORT_NAME = "envitech_reasoning"

SOURCE_URL = (
    "https://huggingface.co/datasets/"
    "kotorii1/EnVi-Tech-Reasoning-SFT"
)

LANGUAGE_PAIR = "en-vi"

DOMAIN = "IT"

DATASET_VERSION = "main"

DOWNLOAD_METHOD = "Hugging Face datasets.load_dataset"

COLLECTION_DATE = pd.Timestamp.now().strftime("%Y-%m-%d")

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / SOURCE_SHORT_NAME
)

RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Source:", SOURCE_NAME)
print("Source URL:", SOURCE_URL)
print("Language pair:", LANGUAGE_PAIR)
print("Domain:", DOMAIN)
print("Dataset version:", DATASET_VERSION)
print("Raw directory:", RAW_DIR)

Source: EnVi-Tech-Reasoning-SFT
Source URL: https://huggingface.co/datasets/kotorii1/EnVi-Tech-Reasoning-SFT
Language pair: en-vi
Domain: IT
Dataset version: main
Raw directory: C:\Users\ADMIN\ENVI-IT-MT\data\raw\envitech_reasoning


In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "kotorii1/EnVi-Tech-Reasoning-SFT"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['en', 'vi', 'category'],
        num_rows: 15115
    })
})


In [6]:
df = dataset["train"].to_pandas()

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Shape: (15115, 3)

Columns:
['en', 'vi', 'category']

Data types:
en          str
vi          str
category    str
dtype: object


In [7]:
print("First 5 rows:")
display(df.head())

print("\nRandom 5 rows:")
display(df.sample(5, random_state=42))

First 5 rows:


,en,vi,category
0,"If the user is logged in, display their profile.","Nếu người dùng đã đăng nhập, hãy hiển thị hồ s...",logic_algo
1,"While the queue is not empty, process the next...","Trong khi hàng đợi chưa rỗng, hãy xử lý mục ti...",logic_algo
2,"For each element in the list, print its value.","Với mỗi phần tử trong danh sách, hãy in giá tr...",logic_algo
3,The boolean expression returns true if both co...,Biểu thức boolean trả về true nếu cả hai điều ...,logic_algo
4,"Check if the input number is positive, negativ...","Kiểm tra xem số đầu vào là số dương, số âm hay...",logic_algo



Random 5 rows:


,en,vi,category
5431,We use DVC for tracking large datasets and mod...,Chúng tôi sử dụng DVC để theo dõi các tập dữ l...,tech_ml_ops
6627,Make a long story short.,Nói tóm lại là...,social_idioms
12714,We are evaluating potential mergers and acquis...,Chúng ta đang đánh giá các thương vụ sáp nhập ...,business_finance
1593,Quantum computing promises to revolutionize co...,Điện toán lượng tử hứa hẹn sẽ cách mạng hóa vi...,tech_ai
5421,Automated unit tests and integration tests are...,Các kiểm thử đơn vị và kiểm thử tích hợp tự độ...,tech_ml_ops


In [8]:
raw_count = len(df)

missing_values = df.isna().sum()

duplicate_count = df.duplicated().sum()

category_distribution = (
    df["category"]
    .value_counts(dropna=False)
    .sort_index()
)

print("Raw count:", raw_count)

print("\nMissing values:")
display(missing_values)

print("\nDuplicate rows:")
print(duplicate_count)

print("\nCategory distribution:")
display(category_distribution)

Raw count: 15115

Missing values:


en          0
vi          0
category    0
dtype: int64


Duplicate rows:
1327

Category distribution:


category
business_email      1050
business_finance    1000
logic_algo          1000
logic_common        1000
logic_english       2050
social_drama         500
social_genz          550
social_idioms        501
tech_ai             1390
tech_coding         2074
tech_hardware       2000
tech_ml_ops         2000
Name: count, dtype: int64

In [9]:
TECH_CATEGORIES = [
    "tech_ai",
    "tech_coding",
    "tech_hardware",
    "tech_ml_ops"
]

df_tech_candidate = (
    df[df["category"].isin(TECH_CATEGORIES)]
    .copy()
)

technology_candidate_count = len(df_tech_candidate)

non_technology_count = (
    raw_count - technology_candidate_count
)

technology_categories = (
    df_tech_candidate["category"]
    .value_counts()
    .sort_index()
)

print("Raw rows:", raw_count)

print(
    "Technology candidate rows:",
    technology_candidate_count
)

print(
    "Non-technology rows:",
    non_technology_count
)

print("\nTechnology categories:")
display(technology_categories)

Raw rows: 15115
Technology candidate rows: 7464
Non-technology rows: 7651

Technology categories:


category
tech_ai          1390
tech_coding      2074
tech_hardware    2000
tech_ml_ops      2000
Name: count, dtype: int64

In [10]:
audit_summary = {
    "source": SOURCE_NAME,
    "raw_count": int(raw_count),
    "candidate_count": int(technology_candidate_count),
    "non_technology_count": int(non_technology_count),
    "missing_values": {
        str(key): int(value)
        for key, value in missing_values.items()
    },
    "duplicate_count": int(duplicate_count),
    "technology_categories": {
        str(key): int(value)
        for key, value in technology_categories.items()
    }
}

audit_summary

{'source': 'EnVi-Tech-Reasoning-SFT',
 'raw_count': 15115,
 'candidate_count': 7464,
 'non_technology_count': 7651,
 'missing_values': {'en': 0, 'vi': 0, 'category': 0},
 'duplicate_count': 1327,
 'technology_categories': {'tech_ai': 1390,
  'tech_coding': 2074,
  'tech_hardware': 2000,
  'tech_ml_ops': 2000}}

In [11]:
raw_jsonl_path = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_raw.jsonl"
)
raw_parquet_path = RAW_DIR / f"{SOURCE_SHORT_NAME}_raw.parquet"
audit_path = RAW_DIR / "audit_summary.json"
metadata_path = RAW_DIR / "metadata.json"

# Phase 01 tạo snapshot RAW một lần; không ghi đè artifact đã có.
phase_01_output_paths = [raw_jsonl_path, raw_parquet_path, audit_path, metadata_path]
existing_outputs = [path for path in phase_01_output_paths if path.exists()]
missing_outputs = [path for path in phase_01_output_paths if not path.exists()]
if existing_outputs and missing_outputs:
    raise RuntimeError(
        "Phát hiện RAW snapshot chưa đầy đủ; không được ghi đè hay tiếp tục. \n"
        f"Existing: {[str(path) for path in existing_outputs]}\n"
        f"Missing: {[str(path) for path in missing_outputs]}"
    )

write_raw_snapshot = not existing_outputs
if write_raw_snapshot:
    print("No existing RAW snapshot found; creating a new immutable snapshot.")
else:
    print("Complete RAW snapshot already exists; preserving it and skipping writes.")

if write_raw_snapshot:
    df.to_json(
        raw_jsonl_path,
        orient="records",
        lines=True,
        force_ascii=False
    )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(raw_jsonl_path)

Complete RAW snapshot already exists; preserving it and skipping writes.
Preserved existing:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\envitech_reasoning\envitech_reasoning_raw.jsonl


In [12]:
raw_parquet_path = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_raw.parquet"
)

if write_raw_snapshot:
    df.to_parquet(
        raw_parquet_path,
        index=False
    )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(raw_parquet_path)

Preserved existing:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\envitech_reasoning\envitech_reasoning_raw.parquet


In [13]:
import json

audit_path = RAW_DIR / "audit_summary.json"

if write_raw_snapshot:
    with open(
        audit_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            audit_summary,
            f,
            ensure_ascii=False,
            indent=2
        )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(audit_path)

Preserved existing:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\envitech_reasoning\audit_summary.json


In [14]:
metadata = {
    "source": SOURCE_NAME,
    "source_url": SOURCE_URL,
    "license": None,
    "version_revision": DATASET_VERSION,
    "collection_date": COLLECTION_DATE,
    "download_method": DOWNLOAD_METHOD,
    "language_pair": LANGUAGE_PAIR,
    "domain": DOMAIN,
    "subcategory": None,
    "raw_count": raw_count,
    "candidate_count": technology_candidate_count,
    "usable_count": None,
    "notes": (
        "Raw dataset collected from source. "
        "Technology candidate count is based on "
        "source category labels. "
        "Full language check, alignment check, "
        "quality/noise assessment, deduplication, "
        "IT subdomain classification and usable-count "
        "confirmation have not yet been completed."
    )
}

metadata_path = RAW_DIR / "metadata.json"

if write_raw_snapshot:
    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            metadata,
            f,
            ensure_ascii=False,
            indent=2
        )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(metadata_path)

Preserved existing:
C:\Users\ADMIN\ENVI-IT-MT\data\raw\envitech_reasoning\metadata.json


In [15]:
expected_files = [
    raw_jsonl_path,
    raw_parquet_path,
    audit_path,
    metadata_path
]

verification_results = {}

for file_path in expected_files:
    verification_results[file_path.name] = file_path.is_file()

print("Final verification:\n")

for file_name, exists in verification_results.items():
    print(
        f"{file_name:40}"
        f"{'OK' if exists else 'MISSING'}"
    )

verification_passed = all(
    verification_results.values()
)

print("\nVerification passed:", verification_passed)

Final verification:

envitech_reasoning_raw.jsonl            OK
envitech_reasoning_raw.parquet          OK
audit_summary.json                      OK
metadata.json                           OK

Verification passed: True


# Data Collection Status

Source:

**EnVi-Tech-Reasoning-SFT**

| Metric | Value |
|---|---:|
| Raw rows | 15,115 |
| Technology candidate rows | 7,464 |
| Non-technology rows | 7,651 |
| Usable IT rows | TBD |

## Completed in this notebook

- [x] Environment checked
- [x] Project root identified
- [x] Source configured
- [x] Dataset downloaded
- [x] Raw schema inspected
- [x] Raw statistics recorded
- [x] Technology candidate identified
- [x] Raw JSONL saved
- [x] Raw Parquet saved
- [x] Audit summary saved
- [x] Metadata saved
- [x] Output files verified

## Not completed in this notebook

- [ ] Full language check
- [ ] Alignment check
- [ ] Data cleaning
- [ ] Deduplication
- [ ] Quality/noise assessment
- [ ] IT subdomain classification
- [ ] Final usable IT count
- [ ] Train / validation / test split

## Interpretation

`raw_count` is not the IT corpus size.

`candidate_count` is not the final usable IT pair count.

For this source:

- `raw_count = 15,115`
- `candidate_count = 7,464`
- `usable_count = TBD`

The 7,464 technology candidates are selected from the source
`category` labels and must pass subsequent audit, cleaning and
IT-filtering steps before being treated as usable IT data.

> Raw data under `data/raw/` must remain unchanged.
>
> This notebook does not produce the final IT corpus.